# Árvores

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx roda o Java com o seu JDK. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Representar uma árvore com um array de pais ou com nós, e achar seus níveis
- Construir uma árvore binária de busca e percorrê-la em ordem, pré-ordem e pós-ordem
- Explicar por que a altura de uma BST decide sua velocidade, e quando ela vira uma lista
- Responder perguntas sobre prefixos com uma trie, ou com palavras ordenadas
- Somar valores subindo por uma árvore com um percurso em pós-ordem

## Em resumo

Uma **árvore** é uma hierarquia: uma **raiz**, e todo outro nó tem exatamente um **pai**. Organogramas, pastas, páginas web e as expressões que um compilador lê são árvores. A **profundidade** de um nó é sua distância até a raiz, e a **altura** da árvore é a profundidade do nó mais fundo. Uma árvore pode morar num **array de pais** (`parent[i]`) ou em **nós** que apontam para os filhos.

**Uma árvore num array de pais:** a profundidade de cada nó vem da do pai, e os filhos são contados numa passada.

In [ ]:
int[] pai = {-1, 0, 0, 1, 1, 2, 4, 4}; // o nó 0 é a raiz
int[] profundidade = new int[pai.length];
int[] filhos = new int[pai.length];
for (int i = 1; i < pai.length; i++) {
    profundidade[i] = profundidade[pai[i]] + 1;
    filhos[pai[i]]++;
}
for (int i = 0; i < pai.length; i++) {
    IO.println("nó " + i + ": profundidade " + profundidade[i] + ", filhos " + filhos[i] + (filhos[i] == 0 ? " (uma folha)" : ""));
}

Uma **árvore binária de busca** (BST) guarda as chaves menores à esquerda de cada nó e as maiores à direita, então uma busca desce por um caminho só: O(altura).

![Uma árvore binária de busca e seu percurso em ordem](bst.pt-br.svg)

Inserções aleatórias dão uma altura de uns 2 log₂ n. Inserções em ordem constroem uma corrente de altura n: uma lista encadeada lenta. Por isso o `TreeMap` do Java se rebalanceia para continuar O(log n).

**Inserções aleatórias contra ordenadas.** Esta célula só acompanha a que profundidade cada chave nova cai; o BST Builder pede a árvore inteira.

In [ ]:
int n = 2000;
for (String ordem : new String[] {"aleatórias", "ordenadas"}) {
    int[] chaves = ordem.equals("aleatórias") ? new Random(5).ints(n, 0, 1_000_000).distinct().toArray() : java.util.stream.IntStream.range(0, n).toArray();
    TreeMap<Integer, Integer> profundidadeDe = new TreeMap<>(); // chave -> sua profundidade na BST
    int altura = 0;
    for (int chave : chaves) {
        // Numa BST, uma chave nova fica pendurada embaixo dos vizinhos mais próximos: o mais fundo dos dois.
        Map.Entry<Integer, Integer> abaixo = profundidadeDe.lowerEntry(chave);
        Map.Entry<Integer, Integer> acima = profundidadeDe.higherEntry(chave);
        int prof = 1 + Math.max(abaixo == null ? 0 : abaixo.getValue(), acima == null ? 0 : acima.getValue());
        profundidadeDe.put(chave, prof);
        altura = Math.max(altura, prof);
    }
    IO.println(String.format("inserções %s de %d chaves: altura %d (log2 n é uns %.0f)", ordem, chaves.length, altura, Math.log(chaves.length) / Math.log(2)));
}

Três **percursos** visitam todos os nós:

- **pré-ordem** (nó, esquerda, direita) copia uma árvore;
- **em ordem** (esquerda, nó, direita) dá as chaves de uma BST ordenadas;
- **pós-ordem** (esquerda, direita, nó) termina os filhos primeiro, e é assim que uma pasta soma seu conteúdo.

Uma **trie** é uma árvore de letras: cada caminho a partir da raiz forma um prefixo, então contar as palavras que começam com "car" leva três passos para baixo.

**Palavras por prefixo num conjunto ordenado:** as palavras que começam com "ca" ficam entre "ca" e "ca{" (`{` vem logo depois de `z`).

In [ ]:
TreeSet<String> palavras = new TreeSet<>(List.of("car", "card", "care", "cart", "cat", "dog", "do", "dot"));
for (String prefixo : new String[] {"ca", "car", "do", "x"}) {
    IO.println(prefixo + " -> " + palavras.subSet(prefixo, prefixo + "{"));
}

Cuidado: recursão numa árvore funda, em forma de corrente, pode estourar a pilha de chamadas.

**Recursão funda:** percorrer uma corrente de 100.000 nós com recursão estoura a pilha de chamadas; um laço não.

In [ ]:
// shared: uma corrente em que o pai do nó i é i - 1
int[] chainParent(int n) {
    int[] parent = new int[n];
    for (int i = 0; i < n; i++) {
        parent[i] = i - 1;
    }
    return parent;
}

int depthRecursive(int[] parent, int node) {
    return parent[node] < 0 ? 0 : 1 + depthRecursive(parent, parent[node]);
}

In [ ]:
int[] parent = chainParent(100_000);
try {
    IO.println("recursivo: " + depthRecursive(parent, 99_999));
} catch (StackOverflowError e) {
    IO.println("recursivo: StackOverflowError");
}
int profundidade = 0;
for (int no = 99_999; parent[no] >= 0; no = parent[no]) {
    profundidade++;
}
IO.println("laço: " + profundidade);

## Leituras

- [Binary Search Tree](https://visualgo.net/en/bst) (VisuAlgo (National University of Singapore), 20 min): Insira suas próprias chaves e depois experimente Search e o percurso em ordem (inorder). Insira 1, 2, 3, 4, 5 numa árvore vazia para vê-la virar uma corrente, e repare no que a árvore AVL do mesmo módulo faz para evitar isso.
- [Introduction to Tree Algorithms](https://usaco.guide/silver/intro-tree) (USACO Guide, 12 min): O vocabulário (raiz, pai, subárvore, profundidade), as três ordens de percurso e o exemplo Subordinates: o tamanho da subárvore de um nó é um mais o dos filhos, a mesma ideia do tamanho total de uma pasta. Escolha Java nas configurações do site para ver código em Java.

## Verifique

1. Que percurso imprime as chaves de uma BST em ordem?
2. O que você obtém inserindo 1, 2, 3, 4, 5 numa BST vazia, e qual é a altura dela?
3. Por que o tamanho de uma pasta precisa de um percurso em pós-ordem?

## Desafios

Resolva os desafios da unidade aqui mesmo: cada célula começa com `// sphinx: challenge …`, e rodá-la roda **todos** os testes do desafio, inclusive os ocultos. Passar conta no seu progresso na barra lateral do Sphinx, como o Submit. A descrição completa, com exemplos e dicas, abre pela barra lateral.

### Organograma · Fácil

In [ ]:
// sphinx: challenge org-chart
void main() {
    int n = Integer.parseInt(IO.readln().trim());
    String[] parts = IO.readln().trim().split(" +");
    int[] manager = new int[n + 1]; // manager[i] para os funcionários 2..n; o funcionário 1 é o CEO
    for (int i = 2; i <= n; i++) {
        manager[i] = Integer.parseInt(parts[i - 2]);
    }

    // TODO: calcule o nível de cada funcionário e depois imprima as quatro linhas
}

### Construtor de BST · Médio

In [ ]:
// sphinx: challenge bst-builder
void main() {
    int n = Integer.parseInt(IO.readln().trim());
    String[] parts = IO.readln().trim().split(" +");
    int[] keys = new int[n];
    for (int i = 0; i < n; i++) {
        keys[i] = Integer.parseInt(parts[i]);
    }

    // TODO: insira as chaves numa árvore binária de busca e depois imprima os percursos, a altura e as folhas
}

### Autocompletar · Médio

In [ ]:
// sphinx: challenge autocomplete
void main() {
    int n = Integer.parseInt(IO.readln().trim());
    String[] words = new String[n];
    for (int i = 0; i < n; i++) {
        words[i] = IO.readln().trim();
    }
    int q = Integer.parseInt(IO.readln().trim());
    for (int i = 0; i < q; i++) {
        String prefix = IO.readln().trim();

        // TODO: imprima quantas palavras começam com prefix, usando uma trie montada com as palavras
    }
}

### Tamanho das pastas · Difícil

In [ ]:
// sphinx: challenge folder-sizes
void main() {
    String[] first = IO.readln().trim().split(" +");
    int n = Integer.parseInt(first[0]);
    long limit = Long.parseLong(first[1]);
    for (int i = 0; i < n; i++) {
        String[] parts = IO.readln().trim().split(" +");
        String path = parts[0];
        long size = Long.parseLong(parts[1]);

        // TODO: adicione o arquivo a uma árvore de pastas
    }

    // TODO: some os totais subindo pela árvore (pós-ordem) e imprima as cinco linhas
}